In [ ]:
import json
import os
import base64
import hashlib
from cryptography.fernet import Fernet, InvalidToken
from getpass import getpass

DATA_FILE = "passwords.enc"
SALT_FILE = "salt.bin"

# KEY DERIVATION 
def derive_key(password, salt):
    key = hashlib.pbkdf2_hmac(
        "sha256",
        password.encode(),
        salt,
        100000
    )
    return base64.urlsafe_b64encode(key)

#  INIT SALT 
def load_or_create_salt():
    if not os.path.exists(SALT_FILE):
        salt = os.urandom(16)
        with open(SALT_FILE, "wb") as f:
            f.write(salt)
    else:
        with open(SALT_FILE, "rb") as f:
            salt = f.read()
    return salt

#  LOAD DATABASE 
def load_data(fernet):
    if not os.path.exists(DATA_FILE):
        return {}

    try:
        with open(DATA_FILE, "rb") as f:
            encrypted = f.read()

        decrypted = fernet.decrypt(encrypted).decode()
        return json.loads(decrypted)

    except InvalidToken:
        print("❌ Wrong master password!")
        exit()

    except Exception:
        return {}

#  SAVE DATABASE 
def save_data(data, fernet):
    encrypted = fernet.encrypt(json.dumps(data, indent=2).encode())
    with open(DATA_FILE, "wb") as f:
        f.write(encrypted)

#  FUNCTIONS 
def add_entry(data):
    site = input("Website: ")
    user = input("Username: ")
    pwd = getpass("Password: ")

    data[site] = {"username": user, "password": pwd}
    print("✅ Saved.")

def view_entry(data):
    site = input("Website to retrieve: ")
    if site in data:
        print("Username:", data[site]["username"])
        print("Password:", data[site]["password"])
    else:
        print("❌ Not found.")

def delete_entry(data):
    site = input("Website to delete: ")
    if site in data:
        del data[site]
        print("🗑 Deleted.")
    else:
        print("❌ Not found.")

def search_entries(data):
    keyword = input("Search keyword: ").lower()
    found = False
    for site in data:
        if keyword in site.lower():
            print(f"{site} -> {data[site]['username']}")
            found = True
    if not found:
        print("No matches.")

#  MAIN 
def main():

    print("=== Secure Password Manager ===")

    salt = load_or_create_salt()
    master = getpass("Enter master password: ")

    key = derive_key(master, salt)
    fernet = Fernet(key)

    data = load_data(fernet)

    while True:
        print("\n1.Add  2.View  3.Delete  4.Search  5.Exit")
        choice = input("Choose: ")

        if choice == "1":
            add_entry(data)
            save_data(data, fernet)

        elif choice == "2":
            view_entry(data)

        elif choice == "3":
            delete_entry(data)
            save_data(data, fernet)

        elif choice == "4":
            search_entries(data)

        elif choice == "5":
            save_data(data, fernet)
            print("Goodbye.")
            break

        else:
            print("Invalid option")

if __name__ == "__main__":
    main()

=== Secure Password Manager ===


Enter master password:  ········



1.Add  2.View  3.Delete  4.Search  5.Exit


Choose:  2
